# Phase 5: site-shift retrain with full replay (Colab T4)

Runs one command end to end on a GPU:

    python -m ppe.loop --config configs/loop/p5_site_shift_colab.yaml

It replays the synthetic dark-site stream (low light, severity 3), mines 200 frames, fine-tunes best.pt with
**all 1,132 training images** as replay (nothing frozen, 20 epochs), then runs the promotion gate on the golden set
and appends the decision to `results/promotions.csv` and every number to `results/results.csv`.

Runtime: GPU (T4). Expect well under an hour. Do not change the config: the recipe was fixed before the run,
and re-running with a different recipe after seeing the gate result would tune on the golden set.

At the end, upload the three files the last cell prints (from your Drive folder `PPE_phase5`) to the project thread.

In [ ]:
!nvidia-smi
BRANCH = "claude/phase5-agentic-loop-yy6qm8"   # change once merged
!git clone --branch {BRANCH} https://github.com/omgawande1523/PPE.git
%cd PPE
!git log --oneline -1

In [ ]:
# Pinned versions. Colab may print dependency warnings for packages this repo does not use.
!pip install -q -r requirements.txt
import ultralytics, torch; print(ultralytics.__version__, torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Downloads the Ultralytics release and refuses to extract it if the sha256 differs.
!python -m ppe.data --download construction-ppe
!python -m ppe.golden --verify golden/golden_v1.csv

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"
!python -m ppe.loop --config configs/loop/p5_site_shift_colab.yaml 2>&1 | grep -v "it/s]" 

In [ ]:
# Promotion log and this run's tables
!python -m ppe.loop --table | tail -n 120

In [ ]:
# Keep the outputs on Drive: the new results rows, the decision, the run's state file, the human-check sample
# and the candidate weights.
import glob, shutil
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/PPE_phase5"
os.makedirs(OUT, exist_ok=True)
run = sorted(glob.glob("runs/loop/p5_site_shift_colab_*"))[-1]
shutil.copy("results/results.csv", f"{OUT}/results_colab.csv")
shutil.copy("results/promotions.csv", f"{OUT}/promotions_colab.csv")
shutil.copy(f"{run}/state.json", f"{OUT}/state_colab.json")
shutil.copytree(f"{run}/human_check", f"{OUT}/human_check", dirs_exist_ok=True)
for w in glob.glob(f"{run}/train/candidate/weights/last.pt") + glob.glob(f"{run}/promoted/*.pt"):
    shutil.copy(w, f"{OUT}/{os.path.basename(os.path.dirname(os.path.dirname(w)))}_{os.path.basename(w)}")
print("Upload these three to the project thread:")
for f in ("results_colab.csv", "promotions_colab.csv", "state_colab.json"):
    print("  ", f"{OUT}/{f}")